# Getting Started with Strands Agents for Financial Services

This lab introduces Strands Agents through the lens of financial services. You will build an AI agent with tools commonly needed by FSI teams — loan calculations, stock lookups, and FX rate checks.

## Overview

In this lab, you will:
- Understand the core concepts of Strands Agents
- Create your first AI agent with built-in tools
- Build custom tools for specific use cases
- Explore conversation history and agent memory
- Learn best practices for agent development


## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

⚠️ **Important**: Enable Nova Pro model access in the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/home#/modelaccess) if you haven't already.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich

In [1]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## What are Strands Agents?

Strands Agents is a Python framework that simplifies the creation of AI agents with tool integration capabilities. Key features include:

- **Simple Agent Creation**: Easy-to-use API for creating AI agents with minimal code
- **Built-in Tools**: Pre-built tools like calculators, web search, and more
- **Custom Tool Support**: Create your own tools with simple Python functions
- **Conversation Memory**: Automatic conversation history management
- **Model Flexibility**: Support for various language models including models in Amazon Bedrock and OpenAI

Strands Agents provides a foundation for building sophisticated AI applications that can interact with external systems and perform complex tasks.


## Building Financial Tools

Let's create three custom tools that a financial services agent would need:

1. **Loan Calculator** — Calculate mortgage/loan repayments
2. **Stock Lookup** — Get current stock prices (ASX/US)
3. **FX Rate** — Currency conversion rates

In [2]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator
import yfinance as yf


@tool
def loan_calculator(principal: float, annual_rate: float, years: int) -> str:
    """Calculate monthly loan/mortgage repayment using standard amortization formula.

    Args:
        principal: Loan amount in dollars
        annual_rate: Annual interest rate as percentage (e.g., 6.2 for 6.2%)
        years: Loan term in years
    """
    monthly_rate = (annual_rate / 100) / 12
    num_payments = years * 12
    if monthly_rate == 0:
        monthly_payment = principal / num_payments
    else:
        monthly_payment = principal * (monthly_rate * (1 + monthly_rate)**num_payments) / ((1 + monthly_rate)**num_payments - 1)
    total_paid = monthly_payment * num_payments
    total_interest = total_paid - principal
    return (
        f"Loan: ${principal:,.2f} at {annual_rate}% over {years} years
"
        f"Monthly repayment: ${monthly_payment:,.2f}
"
        f"Total interest: ${total_interest:,.2f}
"
        f"Total paid: ${total_paid:,.2f}"
    )


@tool
def stock_lookup(ticker: str) -> str:
    """Look up current stock price for ASX or US equities using live market data.

    Args:
        ticker: Stock ticker symbol (e.g., CBA.AX, BHP.AX, AAPL)
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        price = info.get("currentPrice") or info.get("regularMarketPrice", "N/A")
        prev_close = info.get("previousClose", 0)
        name = info.get("shortName", ticker)
        currency = info.get("currency", "")
        if price != "N/A" and prev_close:
            change_pct = ((price - prev_close) / prev_close) * 100
            direction = "▲" if change_pct > 0 else "▼"
            return f"{name} ({ticker}): ${price:.2f} {currency} {direction} {abs(change_pct):.1f}%"
        return f"{name} ({ticker}): ${price} {currency}"
    except Exception as e:
        return f"Error looking up {ticker}: {e}"


@tool
def fx_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> str:
    """Get foreign exchange rate and convert currency amount using live rates.

    Args:
        from_currency: Source currency code (e.g., AUD, USD, GBP)
        to_currency: Target currency code (e.g., USD, AUD, EUR)
        amount: Amount to convert (default 1.0)
    """
    try:
        pair = f"{from_currency}{to_currency}=X"
        data = yf.Ticker(pair)
        rate = data.info.get("regularMarketPrice", None)
        if rate:
            converted = amount * rate
            return f"{amount:,.2f} {from_currency} = {converted:,.2f} {to_currency} (rate: {rate:.4f})"
        return f"Could not fetch rate for {from_currency}/{to_currency}"
    except Exception as e:
        return f"Error: {e}"


print("✅ Financial tools defined: loan_calculator, stock_lookup (live), fx_rate (live)")


✅ FSI tools defined: loan_calculator, stock_lookup, fx_rate


## Creating Your FSI Agent

Now let's create an agent with these financial tools. The agent will decide which tool to use based on your question.

In [3]:
# Create the FSI agent
fsi_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""You are a financial services assistant supporting banking and fintech operations 
    in Australia. You provide concise, accurate 
    financial calculations and market data. Always show your working and cite the tools used.""",
    tools=[loan_calculator, stock_lookup, fx_rate, calculator],
)

fsi_agent("What's the monthly repayment on a $750,000 mortgage at 6.2% over 30 years?")

<thinking> To calculate the monthly repayment on a mortgage, I need to use the loan_calculator tool. The principal amount is $750,000, the annual interest rate is 6.2%, and the loan term is 30 years. </thinking>

Tool #1: loan_calculator
The monthly repayment on a $750,000 mortgage at 6.2% over 30 years is $4,593.52. 

Here's the breakdown:
- Loan amount: $750,000.00
- Annual interest rate: 6.2%
- Loan term: 30 years
- Monthly repayment: $4,593.52
- Total interest paid: $903,666.24
- Total amount paid: $1,653,666.24

This calculation was performed using the loan_calculator tool.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "The monthly repayment on a $750,000 mortgage at 6.2% over 30 years is $4,593.52. \n\nHere's the breakdown:\n- Loan amount: $750,000.00\n- Annual interest rate: 6.2%\n- Loan term: 30 years\n- Monthly repayment: $4,593.52\n- Total interest paid: $903,666.24\n- Total amount paid: $1,653,666.24\n\nThis calculation was performed using the loan_calculator tool."}], 'metadata': {'usage': {'inputTokens': 1972, 'outputTokens': 141, 'totalTokens': 2113}, 'metrics': {'latencyMs': 1453, 'timeToFirstByteMs': 555}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'loan_calculator': ToolMetrics(tool={'toolUseId': 'tooluse_zAveHc5Vbo7NgQEdLTGK4r', 'name': 'loan_calculator', 'input': {'principal': 750000, 'years': 30, 'annual_rate': 6.2}}, call_count=1, success_count=1, error_count=0, total_time=0.0013446807861328125)}, cycle_durations=[1.4302558898925781, 1.5046348571777344], agent_invocations=[AgentInvocati

In [4]:
fsi_agent("What's the current price of CBA and how does it compare to Westpac?")

<thinking> To get the current price of CBA and Westpac, I need to use the stock_lookup tool. The ticker symbols for CBA and Westpac are CBA.AX and WBC.AX, respectively. I will perform two separate lookups to get the current prices and then compare them. </thinking> 
Tool #2: stock_lookup

Tool #3: stock_lookup
The current price of CBA (Commonwealth Bank) is $112.45 AUD, with a 1.2% increase. The current price of Westpac (WBC.AX) is $25.80 AUD, with a 0.5% increase.

Here's the comparison:
- CBA: $112.45 AUD (▲ 1.2%)
- Westpac: $25.80 AUD (▲ 0.5%)

These prices were obtained using the stock_lookup tool.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "The current price of CBA (Commonwealth Bank) is $112.45 AUD, with a 1.2% increase. The current price of Westpac (WBC.AX) is $25.80 AUD, with a 0.5% increase.\n\nHere's the comparison:\n- CBA: $112.45 AUD (▲ 1.2%)\n- Westpac: $25.80 AUD (▲ 0.5%)\n\nThese prices were obtained using the stock_lookup tool."}], 'metadata': {'usage': {'inputTokens': 2309, 'outputTokens': 112, 'totalTokens': 2421}, 'metrics': {'latencyMs': 1114, 'timeToFirstByteMs': 409}}}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'loan_calculator': ToolMetrics(tool={'toolUseId': 'tooluse_zAveHc5Vbo7NgQEdLTGK4r', 'name': 'loan_calculator', 'input': {'principal': 750000, 'years': 30, 'annual_rate': 6.2}}, call_count=1, success_count=1, error_count=0, total_time=0.0013446807861328125), 'stock_lookup': ToolMetrics(tool={'toolUseId': 'tooluse_HUJ58Avv1n4071bVrgTDwj', 'name': 'stock_lookup', 'input': {'ticker': 'WBC.AX'}}, call_cou

In [ ]:
fsi_agent("Convert $1,000,000 AUD to USD and EUR")

## Understanding the Agent Loop

Let's examine how the agent processes requests — which tools it called, what inputs it provided, and what results it received. This visibility is critical for FSI compliance (audit trail).

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {fsi_agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in fsi_agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(
        message["role"], text[-1] if text else "",
        tool_name[-1] if tool_name else "",
        json.dumps(tool_input[-1], indent=2) if tool_input else "",
        (json.dumps(tool_result[-1], indent=2)[:500]) if tool_result else ""
    )

console.print(table)

## Conversation Memory (Within Session)

The agent remembers context within a session. This is useful for follow-up questions — like a real conversation with a financial advisor.

In [ ]:
fsi_agent("If the rate drops to 5.5%, how much would I save per month on that mortgage?")

In [ ]:
fsi_agent("What have we discussed so far?")

## Memory Limitation: New Session = Blank Slate

If we create a new agent instance, it forgets everything. This is a key limitation we'll solve in Lab 06 with AgentCore Memory.

In [ ]:
# New agent = new session = no memory
fresh_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a financial services assistant. Provide concise responses.",
    tools=[loan_calculator, stock_lookup, fx_rate, calculator],
)

fresh_agent("What mortgage rate were we discussing?")

## Summary

In this lab, you:

- ✅ Built custom FSI tools (loan calculator, stock lookup, FX rates)
- ✅ Created an agent that autonomously selects the right tool
- ✅ Examined the agent loop (audit trail for compliance)
- ✅ Explored session-based memory and its limitations

### Next Steps

In the following labs, we'll enhance this agent with:
- **Lab 01**: Code Interpreter for dynamic fraud analysis and risk calculations
- **Lab 02**: Browser automation for regulatory monitoring
- **Lab 04**: Deploy tools as managed services (AgentCore Runtime)
- **Lab 05**: Full observability and audit trails
- **Lab 06**: Persistent memory across sessions (remember client preferences)